[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kvoytova-cyber/museumcanon/blob/main/demo.ipynb)

# museumcanon: русское и советское искусство в MoMA и Tate

**Что делает инструмент.** Берёт открытые данные любого музея и показывает, как в его коллекции и выставках представлена выбранная группа художников: кто, сколько, когда поступало, как (дар или покупка), с кем выставлялось.

- **На входе:** файлы музея (csv / xlsx / json) + YAML-маппинг «какое поле в какой колонке» + YAML с группами художников.
- **На выходе:** сводная таблица, графики поступлений и выставок, сеть совместных выставок, таблицы дарителей и «посредников».

Ноутбук показывает три блока пакета на демо-кейсе (Россия, бывшая империя/СССР, Восточная Европа в MoMA и Tate), а в конце — что тот же код без изменений работает с другой группой и с файлом в другом формате.

## Запуск

**Google Colab:** Runtime → Run all. Первая ячейка клонирует репозиторий и ставит зависимости — остальной ноутбук не отличается от локального запуска.

**Локально:** запускать из корня проекта (там, где лежит этот файл) после `pip install -r requirements.txt`.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/kvoytova-cyber/museumcanon.git"
REPO_DIR = "museumcanon"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    print(f"Colab: репозиторий в {os.getcwd()}, зависимости установлены.")
else:
    print(f"Локальный запуск из {os.getcwd()}.")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd

import museumcanon as mc

pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 110

## 0. Данные

Ссылки лежат в конфигах (`url` у каждой таблицы). Уже скачанное повторно не качается. Коллекция MoMA хранится в Git LFS: если не скачается, положите `Artists.csv` и `Artworks.csv` в `data/raw/` как `moma_artists.csv` и `moma_artworks.csv` вручную — ноутбук продолжит работать и без них, на выставках MoMA и Tate.

In [ ]:
mc.download(["configs/moma.yaml", "configs/tate.yaml"])

## 1. Чтение: файл → объекты

`load_collection` читает конфиг, выбирает ридер по расширению (`CsvReader`, `ExcelReader`, `JsonReader` — наследники `TableReader`) и превращает строки в `Artist`, `Artwork`, `Exhibition`, собранные в `Collection`.

In [ ]:
moma = mc.load_collection("configs/moma.yaml", skip_missing=True)
tate = mc.load_collection("configs/tate.yaml", skip_missing=True)
moma, tate

In [ ]:
gabo_moma = moma.find("Naum Gabo")[0]
gabo_tate = tate.find("Gabo")[0]
print(gabo_moma, "|", gabo_moma.artist_id, gabo_moma.nationality, gabo_moma.birth_country)
print(gabo_tate, "|", gabo_tate.artist_id, gabo_tate.birth_place, gabo_tate.birth_country)

# Artist.__eq__ / __hash__: один человек в двух музеях
print("один и тот же художник:", gabo_moma == gabo_tate)

In [ ]:
# Collection.__add__: два музея -> одна коллекция с общей моделью данных
both = moma + tate
print(both)
print("работ (Collection.__len__):", len(both))
print("Габо есть в объединённой коллекции (__contains__):", gabo_tate in both)
print("художников без дублей между музеями:", len(both.unique_artists()), "из", len(both.artists))

# Artwork.__lt__: сортировка по году поступления
first = sorted(both.artworks_by(gabo_tate))[:3]
[(w.title, w.acquisition_year, w.acquisition_type) for w in first]

## 2. Предобработка и отбор

`Cleaner` применяет секцию `cleaning` из конфига музея: нормализует страны (`Rossiya` → `Russia`), вносит ручные поправки, убирает безымянных «производителей» и исключает ложные срабатывания. Каждый шаг помечен декоратором `@step` и печатает, сколько записей он изменил.

In [ ]:
moma = mc.Cleaner.from_config("configs/moma.yaml").run(moma)
tate = mc.Cleaner.from_config("configs/tate.yaml").run(tate)
both = moma + tate

Группы задаются в `configs/groups_russia.yaml`: страны рождения и национальности. Художник попадает в первую подходящую группу. **Эмигрант** — родился в стране группы, но музей записал другую национальность (Кандинский в MoMA: `French, born Russia`). В Tate национальностей нет, поэтому там эмигрантов определить нельзя.

In [ ]:
grouping = mc.Grouping.from_yaml("configs/groups_russia.yaml")
russia, ex_ussr, east = grouping.selectors

kandinsky = moma.find("Kandinsky")[0]
print(kandinsky, kandinsky.nationality, kandinsky.birth_country)
print("в группе «Россия» (Selector.__contains__):", kandinsky in russia, "| эмигрант:", russia.is_emigre(kandinsky))

# Selector.__or__: объединить группы в одно правило
wide = russia | ex_ussr
chagall = moma.find("Marc Chagall")[0]
print(chagall, chagall.birth_country, "| в «Россия»:", chagall in russia, "| в «Россия + бывш. СССР»:", chagall in wide)

## 3. Визуализация

In [ ]:
viz = mc.Visualizer(both, grouping, start=1920, bin_size=10, out_dir="output")
viz.summary()

### Поступления по десятилетиям

Штриховка выделяет работы одного дарителя (параметр `highlight` — регулярное выражение по credit line). Для MoMA это дар Judith Rothschild Foundation 2001 года: без выделения он выглядит как «пик интереса к русскому искусству в 2000-е».

In [ ]:
viz.plot_acquisitions(highlight="Rothschild");

In [ ]:
# доля от всех поступлений музея за период: так можно сравнивать музеи разного масштаба
viz.plot_acquisitions(share=True);
viz.acquisitions("MoMA", share=True)

### Кто сформировал корпус: дары и покупки, крупнейшие дарители

In [ ]:
viz.plot_acquisition_types(group="Россия");
viz.top_donors("MoMA", "Россия", n=10)

### Выставки MoMA 1929–1989

In [ ]:
viz.plot_exhibitions("Россия");
viz.top_exhibitions("Россия", n=10)

### Сеть совместных выставок

Узлы — художники группы, ребро — не меньше `min_shared` общих выставок. Таблица «посредничества» (betweenness) показывает, через кого связана сеть. Интерактивная версия со всеми группами сохраняется в `output/network.html`.

In [ ]:
viz.plot_network(["Россия"], min_shared=2);
viz.centrality(viz.co_exhibition_graph(["Россия"], min_shared=2), n=12)

In [ ]:
viz.plot_network(grouping.names, min_shared=2, html="network.html")

In [ ]:
from IPython.display import HTML

# HTML(filename=...) встраивает содержимое файла прямо в ячейку вывода.
# IFrame(src="путь к локальному файлу") в Colab часто не работает: песочница вывода
# не видит локальную файловую систему по относительному пути так, как это делает Jupyter.
HTML(filename="output/network.html")

### Те же графики с другими параметрами

Все методы принимают параметры, а настройки визуализатора можно поменять и перестроить график.

In [ ]:
fine = mc.Visualizer(both, grouping, start=1950, end=1995, bin_size=5)
fine.plot_acquisitions(museums=["MoMA"], highlight="Estorick");

## 4. Другой вопрос — тот же код

Меняем только YAML с группами: японские художники в тех же музеях.

In [ ]:
japan = mc.Visualizer(both, mc.Grouping.from_yaml("configs/groups_japan.yaml"))
display(japan.summary())
japan.plot_acquisitions();

## 5. Другой формат — тот же код

`examples/excel_example.xlsx` — Excel с двумя листами и русскими названиями колонок, годы жизни одной строкой. Всё, что нужно, — маппинг в `examples/excel_example.yaml`. Если колонка указана неверно, загрузчик подскажет похожие названия.

In [ ]:
excel = mc.load_collection("examples/excel_example.yaml")
excel = mc.Cleaner.from_config("examples/excel_example.yaml").run(excel)
mc.Visualizer(excel, grouping).summary()

In [ ]:
try:
    mc.load_collection({"name": "тест", "tables": {"artists": {
        "path": "examples/excel_example.xlsx", "read_options": {"sheet_name": "Авторы"},
        "fields": {"artist_id": "инв код автора", "name": "Автор"}}}})
except mc.ConfigError as error:
    print(error)

## Ограничения

- Группа определяется по записи музея: национальность и место рождения могут быть неполными или спорными (украинцы, эмигранты, «Russia» в границах империи). Спорные случаи правятся в секции `cleaning` конфига.
- Художники сопоставляются между музеями по имени и году рождения: «Natalia» и «Natalya» Гончарова не совпадут.
- Данные Tate — срез на 2014 год (с тех пор не обновлялись), выставки MoMA — только 1929–1989.
- Год рождения и год создания извлекаются из строк по первому найденному году: для «c. 1915–20» это 1915.